# Aula 23 — Reprodutibilidade, provenance e zero data leakage

Este laboratório produz uma pequena cadeia de evidência: snapshot canônico, folds por entidade, previsões *out-of-fold*, métricas, hashes e manifesto. Depois, executa contraprovas de leakage.

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/03-machine-learning/notebooks/23-reprodutibilidade-provenance-leakage-laboratorio.ipynb)

## Dependências e protocolo

- Python ≥ 3.11
- NumPy ≥ 1.26
- pandas ≥ 2.1
- scikit-learn ≥ 1.4
- Matplotlib ≥ 3.8

Dados inteiramente sintéticos, seed fixa `20260908`, quatro atendimentos por conta e target constante dentro da conta. A pergunta é: **o modelo generaliza para contas ainda não observadas?** Por isso o split correto é por `account_id`.

In [ ]:
import hashlib
import json
import platform
import re
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("error")
SEED = 20260908
CODE_REF = "ai-lab/ml23-notebook-v1"

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("Seed:", SEED)

## 1. Serialização canônica e identidade

Hash identifica bytes. Portanto, fixamos ordem de linhas/colunas, terminador de linha, formato decimal e JSON com chaves ordenadas. Em produção, `code_ref` deve ser um commit imutável; o valor didático mantém este notebook autossuficiente antes e depois da publicação.

In [ ]:
def sha256_bytes(payload):
    return hashlib.sha256(payload).hexdigest()

def canonical_json_bytes(value):
    return json.dumps(
        value, sort_keys=True, separators=(",", ":"), ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")

def canonical_dataset_bytes(frame):
    ordered = frame.sort_values(["account_id", "visit"], kind="stable")
    ordered = ordered[[
        "account_id", "visit", "score_t0", "usage_t0",
        "post_resolution", "target",
    ]]
    return ordered.to_csv(
        index=False, float_format="%.12g", lineterminator="\n"
    ).encode("utf-8")

assert sha256_bytes(b"abc") == (
    "ba7816bf8f01cfea414140de5dae2223"
    "b00361a396177a9cb410ff61f20015ad"
)
assert canonical_json_bytes({"b": 2, "a": 1}) == b'{"a":1,"b":2}'

## 2. Snapshot sintético documentado

Cada conta possui um risco latente e quatro atendimentos. `score_t0` e `usage_t0` são medidas ruidosas disponíveis no instante de previsão. `post_resolution` é calculada depois do desfecho e existe apenas para a contraprova.

In [ ]:
def generate_dataset(seed, n_accounts=500, visits=4):
    rng = np.random.default_rng(seed)
    latent = rng.normal(size=n_accounts)
    account_target = (latent + rng.normal(0, 0.55, n_accounts) > 0).astype(int)
    repeated_latent = np.repeat(latent, visits)
    target = np.repeat(account_target, visits)
    n = n_accounts * visits
    return pd.DataFrame({
        "account_id": np.repeat([f"A{i:04d}" for i in range(n_accounts)], visits),
        "visit": np.tile(np.arange(visits), n_accounts),
        "score_t0": repeated_latent + rng.normal(0, 2.0, n),
        "usage_t0": 0.3 * repeated_latent + rng.normal(0, 2.0, n),
        "post_resolution": target + rng.normal(0, 0.04, n),
        "target": target,
    })

df = generate_dataset(SEED)
dataset_sha256 = sha256_bytes(canonical_dataset_bytes(df))

assert df.shape == (2000, 6)
assert df["account_id"].nunique() == 500
assert df.groupby("account_id")["target"].nunique().max() == 1
print("Shape:", df.shape)
print("Prevalência:", round(float(df.target.mean()), 6))
print("dataset_sha256:", dataset_sha256)

## 3. Folds são artefatos

Não basta registrar `GroupKFold(5)`: persistimos a atribuição `linha → fold`. O hash muda se a ordem, grupos ou política mudarem.

In [ ]:
def make_group_folds(frame, n_splits=5):
    fold_ids = np.full(len(frame), -1, dtype="<i8")
    splitter = GroupKFold(n_splits=n_splits)
    for fold, (_, valid_idx) in enumerate(
        splitter.split(frame, frame["target"], groups=frame["account_id"])
    ):
        fold_ids[valid_idx] = fold
    assert np.all(fold_ids >= 0)
    return fold_ids

fold_ids = make_group_folds(df)
split_sha256 = sha256_bytes(fold_ids.tobytes())

for fold in np.unique(fold_ids):
    train_groups = set(df.loc[fold_ids != fold, "account_id"])
    valid_groups = set(df.loc[fold_ids == fold, "account_id"])
    assert train_groups.isdisjoint(valid_groups)

print("Linhas por fold:", np.bincount(fold_ids).tolist())
print("split_sha256:", split_sha256)

## 4. Pipeline correto e previsões OOF

O identificador não entra como feature. O scaler é clonado e ajustado somente no treino de cada fold. Guardamos todas as probabilidades OOF e a métrica de cada fold.

In [ ]:
FEATURES_AT_T0 = ["score_t0", "usage_t0"]

def make_correct_pipeline(seed):
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(solver="liblinear", random_state=seed),
    )

def oof_evaluate(frame, assigned_folds, seed):
    probabilities = np.full(len(frame), np.nan, dtype=np.float64)
    metrics = []
    for fold in np.unique(assigned_folds):
        train = assigned_folds != fold
        valid = assigned_folds == fold
        model = clone(make_correct_pipeline(seed))
        model.fit(frame.loc[train, FEATURES_AT_T0], frame.loc[train, "target"])
        probabilities[valid] = model.predict_proba(frame.loc[valid, FEATURES_AT_T0])[:, 1]
        metrics.append(roc_auc_score(frame.loc[valid, "target"], probabilities[valid]))
    assert np.isfinite(probabilities).all()
    return probabilities, np.asarray(metrics)

oof_probability, metrics_by_fold = oof_evaluate(df, fold_ids, SEED)
prediction_sha256 = sha256_bytes(np.round(oof_probability, 12).astype("<f8").tobytes())

assert 0.60 < metrics_by_fold.mean() < 0.80
print("ROC-AUC por fold:", np.round(metrics_by_fold, 6).tolist())
print(f"ROC-AUC: {metrics_by_fold.mean():.6f} ± {metrics_by_fold.std(ddof=1):.6f}")
print("prediction_sha256:", prediction_sha256)

## 5. Manifesto canônico

O ID lógico combina dados, split, configuração e código. Ambiente e métricas contextualizam o run, mas não mudam a identidade do protocolo. Horário pode ser armazenado separadamente como metadado operacional.

In [ ]:
config = {
    "question": "generalizar para contas novas",
    "prediction_time": "t0 do atendimento",
    "target": "desfecho da conta",
    "features_at_t0": FEATURES_AT_T0,
    "splitter": {"class": "GroupKFold", "n_splits": 5, "group": "account_id"},
    "estimator": make_correct_pipeline(SEED).get_params(deep=False),
    "seed": SEED,
    "metric": "roc_auc",
}
# get_params inclui objetos; a identidade usa uma descrição JSON estável.
config["estimator"] = {
    "steps": ["StandardScaler", "LogisticRegression"],
    "solver": "liblinear",
}
config_sha256 = sha256_bytes(canonical_json_bytes(config))
identity = {
    "code_ref": CODE_REF,
    "config_sha256": config_sha256,
    "dataset_sha256": dataset_sha256,
    "split_sha256": split_sha256,
}
experiment_id = sha256_bytes(canonical_json_bytes(identity))

manifest = {
    "experiment_id": experiment_id,
    **identity,
    "environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "platform": platform.system(),
    },
    "metrics": {
        "name": "roc_auc",
        "by_fold": np.round(metrics_by_fold, 12).tolist(),
        "mean": round(float(metrics_by_fold.mean()), 12),
        "sample_std": round(float(metrics_by_fold.std(ddof=1)), 12),
    },
    "prediction_sha256": prediction_sha256,
    "limitations": [
        "dados sintéticos",
        "target constante por conta",
        "generalização avaliada somente para contas novas da mesma população geradora",
    ],
}

def validate_manifest(value):
    required = {
        "experiment_id", "code_ref", "config_sha256", "dataset_sha256",
        "split_sha256", "environment", "metrics", "prediction_sha256", "limitations",
    }
    assert required <= set(value)
    for field in ["experiment_id", "config_sha256", "dataset_sha256", "split_sha256", "prediction_sha256"]:
        assert re.fullmatch(r"[0-9a-f]{64}", value[field])
    assert len(value["metrics"]["by_fold"]) == 5
    assert value["limitations"]

validate_manifest(manifest)
print(json.dumps(manifest, ensure_ascii=False, sort_keys=True, indent=2))

## 6. Repetição controlada

A função abaixo reconstrói dados, folds, previsões e IDs. Com o mesmo protocolo e ambiente, os hashes e as métricas devem coincidir.

In [ ]:
def run_protocol(seed=SEED):
    frame = generate_dataset(seed)
    data_hash = sha256_bytes(canonical_dataset_bytes(frame))
    folds = make_group_folds(frame)
    split_hash = sha256_bytes(folds.astype("<i8").tobytes())
    probabilities, fold_metrics = oof_evaluate(frame, folds, seed)
    pred_hash = sha256_bytes(np.round(probabilities, 12).astype("<f8").tobytes())
    local_config = {**config, "seed": seed}
    cfg_hash = sha256_bytes(canonical_json_bytes(local_config))
    local_identity = {
        "code_ref": CODE_REF,
        "config_sha256": cfg_hash,
        "dataset_sha256": data_hash,
        "split_sha256": split_hash,
    }
    return {
        "experiment_id": sha256_bytes(canonical_json_bytes(local_identity)),
        "dataset_sha256": data_hash,
        "split_sha256": split_hash,
        "prediction_sha256": pred_hash,
        "metrics": fold_metrics,
    }

run_a = run_protocol()
run_b = run_protocol()
assert run_a["experiment_id"] == run_b["experiment_id"]
assert run_a["prediction_sha256"] == run_b["prediction_sha256"]
np.testing.assert_array_equal(run_a["metrics"], run_b["metrics"])
print("IDs idênticos:", run_a["experiment_id"] == run_b["experiment_id"])
print("Previsões idênticas:", run_a["prediction_sha256"] == run_b["prediction_sha256"])

## 7. Teste de mutação do snapshot

Uma única célula numérica muda em $10^{-6}$. A pergunta e o código continuam iguais, mas o snapshot e o ID do experimento precisam mudar.

In [ ]:
mutated = df.copy()
mutated.loc[0, "score_t0"] += 1e-6
mutated_data_hash = sha256_bytes(canonical_dataset_bytes(mutated))
mutated_identity = {**identity, "dataset_sha256": mutated_data_hash}
mutated_experiment_id = sha256_bytes(canonical_json_bytes(mutated_identity))

assert mutated_data_hash != dataset_sha256
assert mutated_experiment_id != experiment_id
assert sha256_bytes(canonical_dataset_bytes(df)) == dataset_sha256
print("Dataset mudou:", mutated_data_hash != dataset_sha256)
print("Experiment ID mudou:", mutated_experiment_id != experiment_id)

## 8. Contraprova 1 — entidade atravessa os folds

No protocolo contaminado, `account_id` vira uma feature e o split é aleatório por linha. Como quase toda conta aparece nos dois lados, o modelo memoriza o desfecho. Repetimos a mesma pipeline com `GroupKFold` para isolar o efeito do split.

In [ ]:
naive_pipeline = make_pipeline(
    ColumnTransformer([
        ("id", OneHotEncoder(handle_unknown="ignore"), ["account_id"]),
        ("num", StandardScaler(), FEATURES_AT_T0),
    ]),
    LogisticRegression(C=100, solver="liblinear", random_state=SEED),
)
row_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
group_cv = GroupKFold(n_splits=5)

p_row = cross_val_predict(
    naive_pipeline, df, df.target, cv=row_cv, method="predict_proba"
)[:, 1]
p_group = cross_val_predict(
    naive_pipeline, df, df.target, groups=df.account_id,
    cv=group_cv, method="predict_proba",
)[:, 1]
auc_row = roc_auc_score(df.target, p_row)
auc_group = roc_auc_score(df.target, p_group)

assert auc_row > 0.98
assert 0.58 < auc_group < 0.80
assert auc_row - auc_group > 0.25
print(f"Split por linha + ID: ROC-AUC={auc_row:.6f}")
print(f"Split por conta + ID desconhecido: ROC-AUC={auc_group:.6f}")
print(f"Otimismo por sobreposição: {auc_row - auc_group:.6f}")

## 9. Contraprova 2 — feature pós-desfecho

Agora o split por conta está correto e o preprocessing continua dentro do fold. Mesmo assim, `post_resolution` praticamente revela o target. Pipeline não conhece a semântica temporal.

In [ ]:
leaked_features = FEATURES_AT_T0 + ["post_resolution"]
leaked_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver="liblinear", random_state=SEED),
)
p_post = cross_val_predict(
    leaked_pipeline, df[leaked_features], df.target,
    groups=df.account_id, cv=group_cv, method="predict_proba",
)[:, 1]
auc_correct = roc_auc_score(df.target, oof_probability)
auc_post = roc_auc_score(df.target, p_post)

assert auc_post > 0.999
assert auc_post - auc_correct > 0.25
assert "post_resolution" not in FEATURES_AT_T0
print(f"Somente features em t0: ROC-AUC={auc_correct:.6f}")
print(f"Com feature pós-desfecho: ROC-AUC={auc_post:.6f}")

In [ ]:
scores = [auc_correct, auc_group, auc_row, auc_post]
labels = ["correto", "grupo + ID", "linha + ID", "pós-desfecho"]
fig, ax = plt.subplots(figsize=(8, 3.5), constrained_layout=True)
bars = ax.bar(labels, scores, color=["#2b8cbe", "#7bccc4", "#fdae6b", "#e34a33"])
ax.set(ylim=(0.45, 1.02), ylabel="ROC-AUC", title="Métricas altas também podem denunciar leakage")
ax.bar_label(bars, fmt="%.3f")
plt.close(fig)
assert len(bars) == 4
print("Figura validada em memória; barras:", len(bars))

## 10. Provenance como grafo

Representamos uma trilha mínima inspirada no W3C PROV. O manifesto é uma entidade gerada pela atividade de avaliação; as previsões derivam do snapshot, dos folds e da configuração.

In [ ]:
prov = [
    ("dataset", "type", "Entity"),
    ("folds", "type", "Entity"),
    ("config", "type", "Entity"),
    ("runner", "type", "SoftwareAgent"),
    ("training", "type", "Activity"),
    ("predictions", "type", "Entity"),
    ("manifest", "type", "Entity"),
    ("training", "used", "dataset"),
    ("training", "used", "folds"),
    ("training", "used", "config"),
    ("training", "wasAssociatedWith", "runner"),
    ("predictions", "wasGeneratedBy", "training"),
    ("predictions", "wasDerivedFrom", "dataset"),
    ("manifest", "wasDerivedFrom", "predictions"),
]

assert ("training", "used", "dataset") in prov
assert ("predictions", "wasGeneratedBy", "training") in prov
assert ("manifest", "wasDerivedFrom", "predictions") in prov
print("Triplas de provenance:", len(prov))
for triple in prov[-7:]:
    print("  ", triple)

## 11. Variabilidade também é resultado

Igualdade exata é apropriada para repetir o mesmo run. Para testar sensibilidade à amostragem, geramos cinco populações sintéticas com seeds diferentes sob o mesmo desenho e reportamos a distribuição.

In [ ]:
seed_scores = []
for seed in range(SEED, SEED + 5):
    frame = generate_dataset(seed)
    folds = make_group_folds(frame)
    _, fold_metrics = oof_evaluate(frame, folds, seed)
    seed_scores.append(float(fold_metrics.mean()))

assert len(set(np.round(seed_scores, 10))) > 1
assert np.std(seed_scores, ddof=1) < 0.08
print("ROC-AUC média por população:", np.round(seed_scores, 6).tolist())
print(f"Entre seeds: {np.mean(seed_scores):.6f} ± {np.std(seed_scores, ddof=1):.6f}")

## Verificações executadas

- SHA-256 conhecido e JSON canônico;
- shape, prevalência e dependência intragrupo documentados;
- zero contas compartilhadas entre treino e validação;
- previsões OOF preenchidas exatamente uma vez;
- manifesto com campos e hashes válidos;
- reexecuções com IDs, métricas e previsões idênticos;
- mutação mínima detectada pelo hash;
- otimismo deliberado por sobreposição de entidades;
- leakage pós-desfecho apesar de pipeline correto;
- cadeia mínima de provenance;
- distribuição em cinco seeds.

## Conclusão

O run correto alcança uma métrica moderada e reconstruível. Os dois experimentos contaminados parecem muito melhores: um reconhece entidades já vistas; o outro lê informação produzida após o desfecho. Todos são executáveis e determinísticos. Somente o primeiro responde à pergunta declarada.

No Gate II, leve esse princípio ao experimento completo: protocolo congelado, baseline, comparação, teste reservado, análise de erros e manifesto ligado à conclusão.